In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from keras.src.utils.dataset_utils import labels_to_dataset
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
# noinspection PyUnresolvedReferences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.regularizers import l2
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SelectKBest, chi2
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

In [3]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
train.head(5)

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [4]:
train.shape

(20758, 18)

In [5]:
train.columns

Index(['id', 'Gender', 'Age', 'Height', 'Weight',
       'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC',
       'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad'],
      dtype='object')

In [6]:
train.drop(columns='id', axis=1, inplace=True)
test.drop(columns='id', axis=1, inplace=True)

In [7]:
train.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,20758.0,23.841804,5.688072,14.00,20.000000,22.815416,26.000000,61.000000
Height,20758.0,1.700245,0.087312,1.45,1.631856,1.700000,1.762887,1.975663
Weight,20758.0,87.887768,26.379443,39.00,66.000000,84.064875,111.600553,165.057269
FCVC,20758.0,2.445908,0.533218,1.00,2.000000,2.393837,3.000000,3.000000
NCP,20758.0,2.761332,0.705375,1.00,3.000000,3.000000,3.000000,4.000000
CH2O,20758.0,2.029418,0.608467,1.00,1.792022,2.000000,2.549617,3.000000
FAF,20758.0,0.981747,0.838302,0.00,0.008013,1.000000,1.587406,3.000000
TUE,20758.0,0.616756,0.602113,0.00,0.000000,0.573887,1.000000,2.000000


In [8]:
train.describe(include='object').T

,count,unique,top,freq
Gender,20758,2,Female,10422
family_history_with_overweight,20758,2,yes,17014
FAVC,20758,2,yes,18982
CAEC,20758,4,Sometimes,17529
SMOKE,20758,2,no,20513
SCC,20758,2,no,20071
CALC,20758,3,Sometimes,15066
MTRANS,20758,5,Public_Transportation,16687
NObeyesdad,20758,7,Obesity_Type_III,4046


In [9]:
print('TRAIN DATA\n')
print(f'{train.isna().sum()}\n\n\n')

print('TEST DATA\n')
print(test.isna().sum())

TRAIN DATA

Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64



TEST DATA

Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                    

In [25]:
X = train


label_encoders = {}
categorical_features = [
    "Gender", "family_history_with_overweight", "FAVC", "CAEC",
    "SMOKE", "SCC", "CALC", "MTRANS", "NObeyesdad"
]
numerical_features = [
    "Age", "Height", "Weight", "CH2O", "FAF", "TUE"
]
        
X.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,1,24.443011,1.699998,81.669950,1,1,2.000000,2.983297,2,0,2.763573,0,0.000000,0.976473,1,3,6
1,0,18.000000,1.560000,57.000000,1,1,2.000000,3.000000,1,0,2.000000,0,1.000000,1.000000,2,0,1
2,0,18.000000,1.711460,50.165754,1,1,1.880534,1.411685,2,0,1.910378,0,0.866045,1.673584,2,3,0
3,0,20.952737,1.710730,131.274851,1,1,3.000000,3.000000,2,0,1.674061,0,1.467863,0.780199,1,3,4
4,1,31.641081,1.914186,93.798055,1,1,2.679664,1.971472,2,0,1.979848,0,1.967973,0.931721,1,3,6


In [26]:
for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))  
    label_encoders[col] = le  

In [28]:
y = X['NObeyesdad']
X = X.drop(['NObeyesdad'], axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
scaler = MinMaxScaler()

X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test[numerical_features] = scaler.transform(X_test[numerical_features])

X_train.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
9958,1,0.063830,0.608755,0.460108,1,1,3.0,3.000000,0,0,0.500000,0,1.000000,0.5,2,0
7841,1,0.184417,0.577155,0.125952,1,1,2.0,4.000000,2,0,0.500000,0,0.666667,0.5,2,3
9293,1,0.166773,0.703620,0.658629,1,1,3.0,2.880817,2,0,0.322669,0,0.246627,0.0,1,3
15209,0,0.574468,0.247307,0.325249,1,1,2.0,3.000000,2,0,0.000000,0,0.000000,0.0,1,0
16515,1,0.191489,0.665826,0.444243,1,0,3.0,3.000000,2,0,1.000000,0,0.666667,0.5,0,3


Tree-Based Feature Importance

In [32]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

feature_importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top_features = feature_importances.nlargest(5).index
print("Top 10 Features by Random Forest Importance:")
print(top_features)

Top 10 Features by Random Forest Importance:
Index(['Weight', 'Age', 'FCVC', 'Height', 'Gender'], dtype='object')


Recursive Feature Elimination (RFE) with Random Forest

In [33]:
rfe_selector = RFE(estimator=rf_model, n_features_to_select=5)
rfe_selector.fit(X_train, y_train)

rfe_selected_features = X_train.columns[rfe_selector.support_]
print("\nFeatures selected by RFE:")
print(rfe_selected_features)


Features selected by RFE:
Index(['Gender', 'Age', 'Height', 'Weight', 'FCVC'], dtype='object')


Chi-Square Feature Selection

In [34]:
X_train_nonneg = X_train.copy()
X_train_nonneg = X_train_nonneg.clip(lower=0)  
chi2_selector = SelectKBest(chi2, k=5)  
chi2_selector.fit(X_train_nonneg, y_train)

chi2_selected_features = X_train.columns[chi2_selector.get_support()]
print("\nFeatures selected by Chi-Square:")
print(chi2_selected_features)


Features selected by Chi-Square:
Index(['Gender', 'Weight', 'family_history_with_overweight', 'SCC', 'MTRANS'], dtype='object')


In [35]:
chi2_selected_features = X_train.columns[chi2_selector.get_support()]
print("\nFeatures selected by Chi-Square:")
print(chi2_selected_features)

# Summary
print("\nCombining these feature selection results for analysis...")
final_features = set(top_features).union(rfe_selected_features).union(chi2_selected_features)
print("Final Selected Features across methods:")
print(final_features)
        



Features selected by Chi-Square:
Index(['Gender', 'Weight', 'family_history_with_overweight', 'SCC', 'MTRANS'], dtype='object')

Combining these feature selection results for analysis...
Final Selected Features across methods:
{'Height', 'family_history_with_overweight', 'Gender', 'Weight', 'MTRANS', 'SCC', 'Age', 'FCVC'}


In [44]:
final_features =  list(final_features)
final_features

['Height',
 'family_history_with_overweight',
 'Gender',
 'Weight',
 'MTRANS',
 'SCC',
 'Age',
 'FCVC']

In [45]:
selected_data = train[final_features]

In [46]:
selected_data.head()

,Height,family_history_with_overweight,Gender,Weight,MTRANS,SCC,Age,FCVC
0,1.699998,1,1,81.669950,3,0,24.443011,2.000000
1,1.560000,1,0,57.000000,0,0,18.000000,2.000000
2,1.711460,1,0,50.165754,3,0,18.000000,1.880534
3,1.710730,1,0,131.274851,3,0,20.952737,3.000000
4,1.914186,1,1,93.798055,3,0,31.641081,2.679664


In [47]:
selected_data.describe().T

,count,mean,std,min,25%,50%,75%,max
Height,20758.0,1.700245,0.087312,1.45,1.631856,1.700000,1.762887,1.975663
family_history_with_overweight,20758.0,0.819636,0.384500,0.00,1.000000,1.000000,1.000000,1.000000
Gender,20758.0,0.497929,0.500008,0.00,0.000000,0.000000,1.000000,1.000000
Weight,20758.0,87.887768,26.379443,39.00,66.000000,84.064875,111.600553,165.057269
MTRANS,20758.0,2.506841,1.148730,0.00,3.000000,3.000000,3.000000,4.000000
SCC,20758.0,0.033096,0.178891,0.00,0.000000,0.000000,0.000000,1.000000
Age,20758.0,23.841804,5.688072,14.00,20.000000,22.815416,26.000000,61.000000
FCVC,20758.0,2.445908,0.533218,1.00,2.000000,2.393837,3.000000,3.000000


In [49]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
encoded_array = encoder.fit_transform(selected_data)

In [52]:
print(encoder.get_feature_names_out())


['Height_1.45' 'Height_1.456346' 'Height_1.463167' ... 'FCVC_2.997951'
 'FCVC_2.998441' 'FCVC_3.0']


In [57]:
scaler = MinMaxScaler()
x_encoded_normalized_train=scaler.fit_transform(selected_data)
x_normalized_df = pd.DataFrame(x_encoded_normalized_train, columns=selected_data.columns)


x_normalized_df.head()

,Height,family_history_with_overweight,Gender,Weight,MTRANS,SCC,Age,FCVC
0,0.475586,1.0,1.0,0.338497,0.75,0.0,0.222192,0.500000
1,0.209260,1.0,0.0,0.142792,0.00,0.0,0.085106,0.500000
2,0.497391,1.0,0.0,0.088577,0.75,0.0,0.085106,0.440267
3,0.496002,1.0,0.0,0.732007,0.75,0.0,0.147931,1.000000
4,0.883049,1.0,1.0,0.434708,0.75,0.0,0.375342,0.839832


In [95]:
def initializeWeights(epsilon_init,L_in, L_out):
    W = np.random.rand(L_out, 1 + L_in) * 2 * epsilon_init - epsilon_init
    return W

In [96]:
def cost_func(nn_params):
        t1 = nn_params[:hidden_layer_size * (input_layer_size + 1)].reshape(
            (hidden_layer_size, input_layer_size + 1))
        t2 = nn_params[hidden_layer_size * (input_layer_size + 1):].reshape(
            (output_layer_size, hidden_layer_size + 1))
        return t1, t2

In [97]:
def relu( z):
    return np.maximum(0, z)

In [98]:
def relu_gradient(z):
    return (z > 0).astype(float)

In [107]:
def softmax( z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [116]:
def compute_cost(y_true, y_pred):
    """Cross-entropy cost function."""
    m = y_true.shape[0]  # Number of samples
    # Avoid division by zero
    epsilon = 1e-5
    cost = -np.sum(y_true * np.log(y_pred + epsilon)) / m
    return cost


In [117]:
def forward_propagation(X, Theta1, Theta2):
    a1 = np.insert(X, 0, 1, axis=1)  
    z2 = np.matmul(a1, Theta1.T)  
    a2 = relu(z2)  
    a2 = np.insert(a2, 0, 1, axis=1) 
    z3 = np.matmul(a2, Theta2.T) 
    a3 = softmax(z3)
    return a1, z2, a2, z3, a3


In [118]:
def backward_propagation(X, y, Theta1, Theta2, a2, a3):
    m = X.shape[0]
    delta3 = a3 - y  
    delta2 = np.matmul(delta3, Theta2[:, 1:].T) * relu_gradient(a2[:, 1:])  

    grad_Theta2 = np.matmul(delta3.T, a2) / m
    grad_Theta1 = np.matmul(delta2.T, np.insert(X, 0, 1, axis=1)) / m

    return grad_Theta1, grad_Theta2


In [119]:
def train_nn(X_train, y_train, input_layer_size, hidden_layer_size, output_layer_size, epochs, alpha=0.01):

    epsilon_init = 0.12
    Theta1 = initializeWeights(epsilon_init, input_layer_size, hidden_layer_size)
    Theta2 = initializeWeights(epsilon_init, hidden_layer_size, output_layer_size)


    y_train = np.eye(output_layer_size)[y_train]

    for epoch in range(epochs):
        a1, z2, a2, z3, a3 = forward_propagation(X_train, Theta1, Theta2)
        cost = compute_cost(y_train, a3)
        grad_Theta1, grad_Theta2 = backward_propagation(X_train, y_train, Theta1, Theta2, a2, a3)
        Theta1 -= alpha * grad_Theta1
        Theta2 -= alpha * grad_Theta2
        print(f"Epoch {epoch + 1}/{epochs}, Cost: {cost}")

    return Theta1, Theta2


In [120]:
# Prepare your data
input_layer_size = len(final_features)  # Features from your dataset
hidden_layer_size = input_layer_size * 2
output_layer_size = len(set(train['NObeyesdad']))

# Train the model
Theta1, Theta2 = train_nn(
    X_train=selected_data,  # Your input data
    y_train=y,  # Your labels
    input_layer_size=input_layer_size,
    hidden_layer_size=hidden_layer_size,
    output_layer_size=output_layer_size,
    epochs=100,  # Number of epochs
    alpha=0.01  # Learning rate
)


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 16 is different from 7)

In [121]:
input_layer_size = len(final_features)
hidden_layer_size = input_layer_size * 2
output_layer_size = len(set(train['NObeyesdad']))
training_history = {'epochs': [], 'costs': [], 'accuracies': []}
lambda_ = 1
epsilon_init = 0.12
Theta1 = initializeWeights(epsilon_init, input_layer_size,hidden_layer_size)
Theta2 = initializeWeights(epsilon_init, hidden_layer_size, output_layer_size)